In [1]:
import numpy as np
import plotly.graph_objects as go
import pandas as pd
from collections import Counter

import numpy as np
from scipy.signal import find_peaks, detrend


# Making the dataframe

In [2]:
energy_df = pd.read_csv('energy_df.csv')
print(energy_df.shape)
print(energy_df.head())
households = energy_df['LCLid'].unique()

(147202070, 3)
       LCLid                 tstp  energy(kWh/hh)
0  MAC000002  2012-10-12 00:30:00             0.0
1  MAC000002  2012-10-12 01:00:00             0.0
2  MAC000002  2012-10-12 01:30:00             0.0
3  MAC000002  2012-10-12 02:00:00             0.0
4  MAC000002  2012-10-12 02:30:00             0.0


In [3]:
example_households = np.random.choice(households, 1)

fig = go.Figure()

df_household = energy_df[energy_df['LCLid'] == example_households[0]]

fig.add_trace(go.Scatter(x=df_household['tstp'], y=df_household['energy(kWh/hh)'], mode='lines', name=example_households[0]))

fig.update_layout(title='Energy consumption for household {}'.format(example_households[0]),
                    xaxis_title='Time',
                    yaxis_title='Energy consumption (kWh/hh)')
fig.show()


In [4]:
energy = df_household['energy(kWh/hh)'].values
print(energy)
print(energy.shape)

[0.    0.131 0.114 ... 0.074 0.069 0.039]
(17435,)


In [5]:
def find_period(data):
    if len(data) == 0:
        return None
    
    # Detrend the data to remove any linear trend
    detrended_data = detrend(data)

    # Autocorrelation to find repeating patterns
    autocorr = np.correlate(detrended_data, detrended_data, mode='full')
    autocorr = autocorr[len(autocorr)//2:]

    # Find peaks in the autocorrelation to determine the period
    peaks, _ = find_peaks(autocorr, height=0)
    periods = np.diff(peaks)  # Differences between consecutive peaks

    # Estimate the period
    if len(periods) > 0:
        estimated_period = np.median(periods)
    else:
        estimated_period = None

    return estimated_period

In [36]:
periods_list = []
counter = 0

for household in households:
    print('Household:', household)
    df_household = energy_df[energy_df['LCLid'] == household]
    energy = df_household['energy(kWh/hh)'].values

    mean_energy = np.mean(energy)
    std_energy = np.std(energy)

    # Get all the energy values that are below the mean minus one standard deviation
    energy_below_mean_std = energy[energy < mean_energy - std_energy]

    if len(energy_below_mean_std) > 0:
        counter += 1
        period = find_period(energy_below_mean_std)

        # Add the period to the list if it is not None
        if period is not None:
            periods_list.append(period)

# Calculate the probability of each period 
period_counts = Counter(periods_list)
total_households = len(example_households)

period_probabilities = {period: count / counter for period, count in period_counts.items()}

# Convert the dictionary to a DataFrame for better visualization
period_prob_df = pd.DataFrame(list(period_probabilities.items()), columns=['Period', 'Probability'])
print(period_prob_df)

Household: MAC000002
Household: MAC000246
Household: MAC000450
Household: MAC001074
Household: MAC003223
Household: MAC003239
Household: MAC003252
Household: MAC003281
Household: MAC003305
Household: MAC003348
Household: MAC003388
Household: MAC003394
Household: MAC003400
Household: MAC003422
Household: MAC003423
Household: MAC003428
Household: MAC003449
Household: MAC003463
Household: MAC003482
Household: MAC003553
Household: MAC003557
Household: MAC003566
Household: MAC003579
Household: MAC003597
Household: MAC003613
Household: MAC003646
Household: MAC003656
Household: MAC003668
Household: MAC003680
Household: MAC003686
Household: MAC003718
Household: MAC003719
Household: MAC003737
Household: MAC003740
Household: MAC003775
Household: MAC003805
Household: MAC003826
Household: MAC003844
Household: MAC003851
Household: MAC003856
Household: MAC003863
Household: MAC003874
Household: MAC004034
Household: MAC004179
Household: MAC004247
Household: MAC004319
Household: MAC004387
Household: MA

In [37]:
period_prob_df.to_csv('../../Data_processed/period_prob_df.csv', index=False)

In [8]:
period_prob_df = pd.read_csv('../../Data_processed/period_prob_df.csv')
# Sort the DataFrame by the period
period_prob_df = period_prob_df.sort_values(by='Probability', ascending=False)
print(period_prob_df)

    Period  Probability
0      3.0     0.562092
3      4.0     0.167756
1      2.0     0.040305
4      5.0     0.031046
5      6.0     0.009804
2      2.5     0.007625
6      4.5     0.007625
7      3.5     0.006536
10     7.0     0.005447
9      8.0     0.004902
8     12.5     0.001089
13     9.0     0.001089
15    10.0     0.001089
11    11.0     0.000545
12     5.5     0.000545
14    13.0     0.000545
16    12.0     0.000545
17    15.0     0.000545
18    14.0     0.000545
19     6.5     0.000545


In [9]:
# Round the period to the nearest integer
period_prob_df['Period'] = period_prob_df['Period'].round().astype(int)
print(period_prob_df)

    Period  Probability
0        3     0.562092
3        4     0.167756
1        2     0.040305
4        5     0.031046
5        6     0.009804
2        2     0.007625
6        4     0.007625
7        4     0.006536
10       7     0.005447
9        8     0.004902
8       12     0.001089
13       9     0.001089
15      10     0.001089
11      11     0.000545
12       6     0.000545
14      13     0.000545
16      12     0.000545
17      15     0.000545
18      14     0.000545
19       6     0.000545


In [10]:
# Add the same period's probability to the same period
period_prob_df = period_prob_df.groupby('Period')['Probability'].sum().reset_index()
print(period_prob_df)


    Period  Probability
0        2     0.047930
1        3     0.562092
2        4     0.181917
3        5     0.031046
4        6     0.010893
5        7     0.005447
6        8     0.004902
7        9     0.001089
8       10     0.001089
9       11     0.000545
10      12     0.001634
11      13     0.000545
12      14     0.000545
13      15     0.000545


In [12]:
# Remove periods with a probability less than 0.001
period_prob_df = period_prob_df[period_prob_df['Probability'] >= 0.001]
# Recalculate the total probability
total_prob = period_prob_df['Probability'].sum()
# Normalize the probabilities
period_prob_df['Probability'] = period_prob_df['Probability'] / total_prob
# Sort the DataFrame by the period
period_prob_df = period_prob_df.sort_values(by='Probability', ascending=False)
print(period_prob_df) 

    Period  Probability
1        3     0.662813
2        4     0.214515
0        2     0.056519
3        5     0.036609
4        6     0.012845
5        7     0.006423
6        8     0.005780
10      12     0.001927
7        9     0.001285
8       10     0.001285


In [13]:
period_prob_df.to_csv('../../Data_processed/period_prob_df.csv', index=False)